# Simple integrated AIA-to-GOES regression

The core calculation in this notebook is intentionally simple. For each calibrated AIA image, sum every wavelength over all pixels and fit nonnegative channel weights to the paired GOES/XRS-B value:

$$\widehat F_{GOES}=\sum_c w_c S_c,\qquad S_c=\sum_{x,y} I_c(x,y),\qquad w_c\geq0.$$

Two residual losses are compared:

- L2: ordinary nonnegative least squares;
- L1: nonnegative least absolute deviations, meaning MAE residual loss rather than Lasso coefficient regularization.

The fitted weights can then be applied to each spatial patch. Normalizing that patch proxy makes a spatial prior whose absolute units do not matter. No DEM is estimated. A final optional section plots the temperature-response curves only as a physical consistency check; those curves do not participate in the empirical regression.

In [ ]:
from pathlib import Path
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
REPO_ROOT = next((path for path in (cwd, *cwd.parents) if (path / 'forecasting').is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError('Run this notebook from inside the FOXES repository')
sys.path.insert(0, str(REPO_ROOT))

from analysis.sxr_response_calibration import (
    AIA_WAVELENGTHS,
    ensure_aia_response,
    ensure_goes_response,
    fit_integrated_weights,
    load_aia_temperature_response,
    load_goes_temperature_response,
    load_integrated_sample,
    normalized_weights,
    patch_proxy_map,
    proxy_values,
    score_proxy,
)

SEED = 42
TRAIN_SAMPLES = int(os.environ.get('FOXES_CALIBRATION_TRAIN_SAMPLES', 512))
VAL_SAMPLES = int(os.environ.get('FOXES_CALIBRATION_VAL_SAMPLES', 256))
AIA_ROOT = Path(os.environ.get('FOXES_CALIBRATED_AIA_ROOT', '/data/AIA_processed'))
SXR_ROOT = Path(os.environ.get('FOXES_SXR_ROOT', '/data/SXR_processed'))
CHANNELS = AIA_WAVELENGTHS
PATCH_SIZE = 8
OUTPUT_DIR = REPO_ROOT / 'notebooks' / 'outputs' / 'sxr_response'
CACHE_DIR = REPO_ROOT / 'notebooks' / '.cache' / 'sxr_response'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Calibrated AIA:', AIA_ROOT)
print('Paired GOES:', SXR_ROOT)
print('Samples:', TRAIN_SAMPLES, 'train /', VAL_SAMPLES, 'validation')

## 1. Load a small paired sample

Use calibrated, nonnegative AIA DN/s arrays. The loader rejects the already normalized FOXES_OG arrays near [-1, 1].

In [ ]:
train_frame = load_integrated_sample(
    AIA_ROOT, SXR_ROOT, 'train', TRAIN_SAMPLES, seed=SEED, aggregation='sum'
)
val_frame = load_integrated_sample(
    AIA_ROOT, SXR_ROOT, 'val', VAL_SAMPLES, seed=SEED + 1, aggregation='sum'
)
print('Loaded', len(train_frame), 'training and', len(val_frame), 'validation pairs')

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(np.log10(np.clip(train_frame['goes_xrsb_w_m2'], 1e-10, None)), bins=35, alpha=0.7, label='train')
ax.hist(np.log10(np.clip(val_frame['goes_xrsb_w_m2'], 1e-10, None)), bins=35, alpha=0.7, label='validation')
ax.set(xlabel='log10 GOES XRS-B [W m$^{-2}$]', ylabel='samples', title='Target coverage')
ax.legend()
plt.show()

## 2. Fit L2 and L1 regressions

There is no intercept. An intercept may improve global prediction, but it would have no physically justified patch location when constructing the spatial prior.

In [ ]:
weights = {
    'least_squares_l2': fit_integrated_weights(train_frame, CHANNELS, loss='l2'),
    'absolute_error_l1': fit_integrated_weights(train_frame, CHANNELS, loss='l1'),
}

weight_table = pd.concat(
    {
        name: pd.DataFrame({
            'coefficient': fitted.reindex(CHANNELS, fill_value=0.0),
            'fraction': normalized_weights(fitted).reindex(CHANNELS, fill_value=0.0),
        })
        for name, fitted in weights.items()
    },
    axis=1,
)
weight_table.index.name = 'wavelength_A'
display(weight_table)

metric_rows = []
for name, fitted in weights.items():
    for split, frame in [('train', train_frame), ('val', val_frame)]:
        metric_rows.append({'method': name, 'split': split, **score_proxy(frame, fitted)})
metrics = pd.DataFrame(metric_rows).set_index(['method', 'split'])
display(metrics.style.format({
    'mae_w_m2': '{:.3e}', 'rmse_w_m2': '{:.3e}',
    'mae_dex': '{:.3f}', 'rmse_dex': '{:.3f}',
    'pearson_log': '{:.3f}', 'spearman': '{:.3f}',
}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fractions = pd.DataFrame({name: normalized_weights(fitted) for name, fitted in weights.items()}).fillna(0.0)
fractions.plot.bar(ax=axes[0])
axes[0].set(xlabel='AIA wavelength [Å]', ylabel='coefficient fraction', title='Learned channel weights')

for name, fitted in weights.items():
    prediction = proxy_values(val_frame, fitted)
    axes[1].scatter(val_frame['goes_xrsb_w_m2'], prediction, s=13, alpha=0.5, label=name)
upper = max(1e-4, float(val_frame['goes_xrsb_w_m2'].max()))
limits = (1e-9, upper)
axes[1].plot(limits, limits, color='black', linestyle='--', linewidth=1)
axes[1].set(xscale='log', yscale='log', xlim=limits, ylim=limits, xlabel='observed GOES [W m$^{-2}$]', ylabel='regression prediction [W m$^{-2}$]', title='Held-out integrated prediction')
axes[1].legend()
fig.tight_layout()
plt.show()

## 3. Apply the same weights spatially

For each method, the weighted intensity is summed inside every 8-by-8-pixel model patch and normalized over the image. This is the proposed prior map.

In [ ]:
example_names = val_frame.sort_values('goes_xrsb_w_m2').iloc[
    np.linspace(0, len(val_frame) - 1, 3, dtype=int)
]['filename'].tolist()
fig, axes = plt.subplots(len(example_names), 4, figsize=(14, 4 * len(example_names)))
for row, filename in enumerate(example_names):
    image = np.asarray(np.load(AIA_ROOT / 'val' / filename, allow_pickle=False), dtype=np.float64)
    for column, channel in enumerate((94, 131)):
        channel_image = np.clip(image[AIA_WAVELENGTHS.index(channel)], 0.0, None)
        axes[row, column].imshow(np.arcsinh(channel_image), origin='lower', cmap='magma')
        axes[row, column].set_title(f'{filename[:-4]} | {channel} Å')
    for column, (name, fitted) in enumerate(weights.items(), start=2):
        spatial_proxy = patch_proxy_map(image, fitted, patch_size=PATCH_SIZE)
        axes[row, column].imshow(spatial_proxy, origin='lower', cmap='inferno')
        axes[row, column].set_title(name + ' | patch proxy')
    for axis in axes[row]:
        axis.set_xticks([])
        axis.set_yticks([])
fig.tight_layout()
plt.show()

## 4. Optional response-function check

This section does not refit the weights. It asks whether each empirically learned AIA mixture has a temperature dependence resembling GOES/XRS-B. A poor match is possible even when the integrated regression works, because real images contain multithermal DEM distributions and correlated channel intensities. The AIA and GOES curves are both pinned to CHIANTI v9 for this comparison. Sources: [demregpy AIA responses](https://demregpy.readthedocs.io/en/latest/generated/gallery/aia/plot_aia_response.html) and the [SolarSoft GOES tables](https://soho.nascom.nasa.gov/solarsoft/gen/idl/synoptic/goes/).

In [ ]:
aia_path = ensure_aia_response(CACHE_DIR)
goes_path = ensure_goes_response(CACHE_DIR, response_version='chianti_v9')
response_channels, aia_logt, aia_response = load_aia_temperature_response(aia_path)
goes_logt, goes_response, goes_metadata = load_goes_temperature_response(goes_path, satellite=15)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(goes_logt, goes_response / goes_response.max(), color='black', linewidth=3, label='GOES-15 XRS-B')
for name, fitted in weights.items():
    channel_weights = fitted.reindex(response_channels, fill_value=0.0).to_numpy()
    empirical_response = aia_response @ channel_weights
    if empirical_response.max() > 0:
        empirical_response = empirical_response / empirical_response.max()
    ax.plot(aia_logt, empirical_response, linewidth=2, label=name + ' AIA mixture')
ax.set(yscale='log', ylim=(1e-7, 2), xlim=(5.5, 8.2), xlabel='log10 temperature [K]', ylabel='response / maximum', title='Physical response check only')
ax.legend()
plt.show()
display(pd.Series(goes_metadata, name='value').to_frame())

In [ ]:
weight_table.to_csv(OUTPUT_DIR / 'integrated_regression_weights.csv')
metrics.reset_index().to_csv(OUTPUT_DIR / 'integrated_regression_metrics.csv', index=False)
manifest = {
    'channels': list(CHANNELS),
    'aggregation': 'sum',
    'intercept': False,
    'train_samples': len(train_frame),
    'validation_samples': len(val_frame),
    'seed': SEED,
    'aia_root': str(AIA_ROOT),
    'sxr_root': str(SXR_ROOT),
    'response_check': {'aia_file': str(aia_path), 'goes_file': str(goes_path), **goes_metadata},
}
with (OUTPUT_DIR / 'integrated_regression_manifest.json').open('w') as stream:
    json.dump(manifest, stream, indent=2)
print('Saved results to', OUTPUT_DIR)